In [1]:
from unet import unet_model

In [14]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = unet_model.UNet(
    n_channels=3,   # RGB-like input
    n_classes=1,    # scalar field output
    bilinear=False,
).to(device)


# One fake 64x64 RGB image
x = torch.rand(
    1, 3, 64, 64,
    device=device,
)

model.eval()

with torch.no_grad():
    output = model(x)


print("Device:", device)
print("Input: ", x.shape)
print("Output:", output.shape)

Device: cpu
Input:  torch.Size([1, 3, 64, 64])
Output: torch.Size([1, 1, 64, 64])


In [15]:
from unet.hexunet_parts import HexConv2d
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

class HexCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = HexConv2d(1, 8)
        self.conv2 = HexConv2d(8, 16)
        self.conv3 = HexConv2d(16, 1)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)

        return x

In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = HexCNN().to(device)

x = torch.rand(
    1,      # batch
    1,      # channel
    64, 64, # axial q/r storage
    device=device,
)

with torch.no_grad():
    y = model(x)

print("Input: ", x.shape)
print("Output:", y.shape)

Input:  torch.Size([1, 1, 64, 64])
Output: torch.Size([1, 1, 64, 64])


In [18]:
from torch.utils.data import TensorDataset, DataLoader


def generate_hex_dataset(
    n_samples=1000,
    height=64,
    width=64,
):
    X = torch.rand(
        n_samples,
        1,
        height,
        width,
    )

    kernel = torch.tensor([
        [0., 1., 1.],
        [1., 1., 1.],
        [1., 1., 0.],
    ]).reshape(1, 1, 3, 3)

    y = F.conv2d(
        X,
        kernel,
        padding=1,
    )

    return X, y


X, y = generate_hex_dataset()

dataset = TensorDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)


model = HexCNN()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

loss_fn = torch.nn.MSELoss()


for epoch in range(20):

    total_loss = 0.0

    model.train()

    for X_batch, y_batch in loader:

        pred = model(X_batch)

        loss = loss_fn(
            pred,
            y_batch,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    print(
        epoch,
        average_loss,
    )


model.eval()

with torch.no_grad():
    pred = model(X[:1])


print("Actual:")
print(
    y[0, 0, 30:35, 30:35]
)

print("Predicted:")
print(
    pred[0, 0, 30:35, 30:35]
)

0 9.572370186448097
1 1.126388511620462
2 0.33199517149478197
3 0.28962923772633076
4 0.24248451087623835
5 0.18695782497525215
6 0.13910905038937926
7 0.10824423376470804
8 0.08822487806901336
9 0.07212788122706115
10 0.05836037022527307
11 0.04750125715509057
12 0.039555404568091035
13 0.03381967649329454
14 0.02960980380885303
15 0.02632214198820293
16 0.023557970649562776
17 0.020967765129171312
18 0.018461785977706313
19 0.016100246837595478
Actual:
tensor([[3.4080, 4.2750, 4.0589, 3.3387, 2.6577],
        [4.3294, 4.8034, 4.4777, 4.1663, 4.4185],
        [4.0955, 4.3969, 3.9205, 3.5060, 3.6832],
        [4.1308, 2.2799, 2.2344, 2.8611, 1.6963],
        [2.4425, 2.6282, 2.2998, 2.2474, 2.6123]])
Predicted:
tensor([[3.4432, 4.2842, 4.0516, 3.5579, 2.7997],
        [4.1505, 4.9482, 4.6052, 4.2816, 4.5617],
        [4.2568, 4.1933, 3.7897, 3.5628, 3.6930],
        [3.7609, 2.4009, 2.3855, 2.8152, 1.7740],
        [2.7911, 2.6269, 2.3199, 2.3710, 2.5610]])


In [19]:
X

tensor([[[[0.6217, 0.1644, 0.8656,  ..., 0.9300, 0.7077, 0.6065],
          [0.5090, 0.5027, 0.6958,  ..., 0.0897, 0.9712, 0.8894],
          [0.7512, 0.7629, 0.8051,  ..., 0.1453, 0.4594, 0.6966],
          ...,
          [0.4035, 0.2296, 0.5861,  ..., 0.4847, 0.9831, 0.9132],
          [0.6556, 0.0387, 0.3105,  ..., 0.3621, 0.7594, 0.3074],
          [0.6239, 0.2111, 0.3031,  ..., 0.4989, 0.8664, 0.8681]]],


        [[[0.2582, 0.7957, 0.7083,  ..., 0.6166, 0.5815, 0.5696],
          [0.2793, 0.9442, 0.3897,  ..., 0.4142, 0.1280, 0.4587],
          [0.3343, 0.1406, 0.9416,  ..., 0.5804, 0.0985, 0.9363],
          ...,
          [0.0965, 0.7072, 0.3030,  ..., 0.1094, 0.6706, 0.7209],
          [0.9840, 0.2595, 0.7815,  ..., 0.9317, 0.4201, 0.8455],
          [0.9962, 0.9843, 0.8461,  ..., 0.4402, 0.6553, 0.3821]]],


        [[[0.1347, 0.5120, 0.8914,  ..., 0.1816, 0.3422, 0.0069],
          [0.0565, 0.2474, 0.3816,  ..., 0.0980, 0.6309, 0.9939],
          [0.4140, 0.3106, 0.7374,  ..

In [21]:
import torch
import torch.nn.functional as F


size = 7
center = 3

x = torch.zeros(
    1,
    1,
    size,
    size,
)

x[0, 0, center, center] = 1.0


kernel = torch.tensor([
    [0., 1., 0.],
    [1., 1., 1.],
    [0., 1., 0.],
]).reshape(1, 1, 3, 3)


y = F.conv2d(
    x,
    kernel,
    padding=1,
)


print("Input:")
print(x[0, 0])

print("\nOutput:")
print(y[0, 0])


active = torch.nonzero(
    y[0, 0],
    as_tuple=False,
)

print("\nActive output cells:")
print(active)

Input:
tensor([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]])

Output:
tensor([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 0., 0., 0.],
        [0., 0., 1., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]])

Active output cells:
tensor([[2, 2],
        [2, 3],
        [3, 2],
        [3, 3],
        [3, 4],
        [4, 3],
        [4, 4]])
